In [6]:
MODEL_ID = 'moonshotai/kimi-k2-instruct-0905'
EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
import os
from dotenv import load_dotenv
load_dotenv("../keys.env")
assert os.environ["GROQ_API_KEY"].startswith("gsk"), \
  "Please specify the GROQ_API_KEY access token in keys.env file"
assert os.environ["HF_TOKEN"].startswith("hf"), \
  "Please specifuy HF_TOKEN environment variable in keys.env file"  

In [7]:
import sys
sys.path.append('../06_basic_rag')
import gutenberg_text_loader as gtl
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s -%(message)s')
logger = logging.getLogger(__name__)

In [8]:
!ls ./.cache vector_index

./.cache:
pg46976_1b16d8525c.txt

vector_index:
default__vector_store.json  graph_store.json	      index_store.json
docstore.json		    image__vector_store.json


In [9]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import Document

INDEX_DIR = 'vector_index'
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_ID)

Settings.chunk_size = 1024
Settings.chunk_overlap = 20
TOP_K = 2

if os.path.isdir(INDEX_DIR):
    print("Loading in already created index")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_DIR)
    index = load_index_from_storage(storage_context)
else:
    gs = gtl.GutenbergSource()
    doc = gs.load_from_url("https://www.gutenberg.org/cache/epub/46976/pg46976.txt")
    # Read all files in ./cache
    documents = SimpleDirectoryReader(input_dir='./.cache', required_exts=['.txt'], exclude_hidden=False).load_data()
    # creates a vector db
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=INDEX_DIR)


2026-04-08 17:07:46,727 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


2026-04-08 17:07:50,241 - INFO - 1 prompt is loaded, with the key: query


Loading in already created index


2026-04-08 17:07:51,325 - INFO - Loading all indices.


In [ ]:
from llama_index.llms.groq import Groq
from llama_index.core.query_engine import RetrieverQueryEngine

llm = Groq(model=MODEL_ID, api_key=os.environ['GROQ_API_KEY'])

query_engine = RetrieverQueryEngine.from_args(
    retriever=index.as_retriever(similarity_top_k=TOP_K), llm=llm,
)

def semantic_rag(question):
    response = query_engine.query(question)
    response = {"answer": str(response),
                "source_nodes": response.source_nodes
                }
    print(response['answer'])

    for node in response['source_nodes']:
        print(node)
    return response

semantic_rag("How did Alexander treat the people of the places he conquered?")